# Roster changes: trades, signings, waivers

Run this on the Mac mini against the live database whenever a player changes team mid-season.

**What it changes**
- `plyrLogs`: removes the player's *unplayed* schedule rows (`min IS NULL`) on his old team from the effective date on, and adds rows for his new team's games from the effective date or today, whichever is later. Games already played get their real rows from the box-score load, never an empty placeholder.
- `rosters`: closes the old team's entry (`endDate` = day before) and opens one for the new team.

**What it never touches:** any game the player actually played (`min` is not NULL). The delete statement itself requires `min IS NULL`, and the apply step counts each player's played games before and after, rolling back if the count moves.

**Steps:** run the cells top to bottom. Nothing is written until `APPLY = True` in the apply cell. To undo a change, see the last section.

In [ ]:
import os
import sqlite3
import datetime as dt
import pandas as pd
from nba.NBAbase import base

DB_PATH = os.environ.get('NBA_DB_PATH', base().db)
SNAPSHOT = os.path.join(os.path.dirname(DB_PATH), 'nba_before_roster_change.db')
SEASON = '2026-27'
TODAY = dt.date.today().isoformat()
conn = sqlite3.connect(DB_PATH)
print('database: {}'.format(DB_PATH))

## 1. Snapshot
One file, overwritten each run: the state right before this change. Run this every time before applying.

In [ ]:
snap = sqlite3.connect(SNAPSHOT)
conn.backup(snap)
snap.close()
print('snapshot written: {} ({:,.0f} MB)'.format(SNAPSHOT, os.path.getsize(SNAPSHOT) / 1e6))

## 2. Enter the changes
- `effective_date`: the first game date the player is with the new team (`YYYY-MM-DD`).
- `new_team`: team abbreviation as in the `teams` table (for example `GSW`); `None` for a waiver or release.
- A signing is a row whose player has no open roster entry.
- A player not yet in `players` must be added first (`NBAetl.etl().get_player_info(player_id)` and insert).

In [ ]:
changes = pd.DataFrame([
    # {'name': 'Player Name', 'new_team': 'GSW', 'effective_date': '2026-11-15'},
])
changes

## 3. Resolve players and teams

In [ ]:
players = pd.read_sql('SELECT player_id, name FROM players', conn)
teams = pd.read_sql('SELECT team_id, teamAbrv FROM teams', conn)
open_roster = pd.read_sql("SELECT teamId AS old_team_id, playerId AS player_id, seasonYear, startDate FROM rosters WHERE endDate IS NULL", conn)

resolved = changes.merge(players, on='name', how='left')
resolved = resolved.merge(teams.rename(columns={'team_id': 'new_team_id', 'teamAbrv': 'new_team'}), on='new_team', how='left')
resolved = resolved.merge(open_roster, on='player_id', how='left')
resolved['old_team'] = resolved.old_team_id.map(teams.set_index('team_id').teamAbrv)

problems = []
for r in resolved.itertuples():
    if pd.isna(r.player_id):
        problems.append('{}: not found in players (check spelling or add the player first)'.format(r.name))
    if r.new_team is not None and pd.isna(r.new_team_id):
        problems.append('{}: unknown team abbreviation {}'.format(r.name, r.new_team))
    if pd.isna(r.old_team_id) and r.new_team is None:
        problems.append('{}: no open roster entry and no new team; nothing to do'.format(r.name))
    if r.old_team_id == r.new_team_id:
        problems.append('{}: already on {}'.format(r.name, r.new_team))
if resolved.player_id.duplicated().any():
    problems.append('a player appears more than once; enter one change per player per run')
print('\n'.join(problems) if problems else 'all changes resolved')
resolved[['name', 'player_id', 'old_team', 'new_team', 'effective_date']]

## 4. Preview
Shows what will be removed and added for each player. A player with **played** games on or after the effective date blocks the run: the date is wrong, or the move already happened.

In [ ]:
preview = []
for r in resolved.itertuples():
    played_after = conn.execute(
        "SELECT COUNT(*) FROM plyrLogs WHERE player_id = ? AND game_date >= ? AND min IS NOT NULL",
        (r.player_id, r.effective_date)).fetchone()[0]
    remove = conn.execute(
        "SELECT COUNT(*) FROM plyrLogs WHERE player_id = ? AND team_id = ? AND game_date >= ? AND min IS NULL",
        (r.player_id, r.old_team_id, r.effective_date)).fetchone()[0] if not pd.isna(r.old_team_id) else 0
    add = conn.execute(
        '''SELECT COUNT(*) FROM teamLog t
           WHERE t.team_id = ? AND t.game_date >= ?
             AND NOT EXISTS (SELECT 1 FROM plyrLogs p WHERE p.player_id = ? AND p.game_id = t.game_id)''',
        (r.new_team_id, max(r.effective_date, TODAY), r.player_id)).fetchone()[0] if not pd.isna(r.new_team_id) else 0
    preview.append({'name': r.name, 'old_team': r.old_team, 'new_team': r.new_team, 'effective_date': r.effective_date,
                    'played_games_on_or_after_date': played_after, 'placeholder_rows_to_remove': remove, 'schedule_rows_to_add': add})
preview = pd.DataFrame(preview)
blocked = preview[preview.played_games_on_or_after_date > 0]
if len(blocked):
    print('BLOCKED: these players have played games on or after the effective date. Fix the date before applying.')
preview

## 5. Apply
Set `APPLY = True` and run. Everything happens in one transaction: if any player's played-game count changes, or anything errors, nothing is saved.

In [ ]:
APPLY = False

if not APPLY:
    print('Dry run: set APPLY = True to write these changes.')
elif problems or len(blocked) or resolved.empty:
    print('Not applied: fix the problems or blocked players above first.')
else:
    ids = resolved.player_id.tolist()
    marks = ','.join('?' * len(ids))
    played_q = 'SELECT player_id, COUNT(*) FROM plyrLogs WHERE player_id IN ({}) AND min IS NOT NULL GROUP BY player_id'.format(marks)
    before = dict(conn.execute(played_q, ids).fetchall())
    pcols = [c[1] for c in conn.execute('PRAGMA table_info(plyrLogs)')]
    try:
        with conn:
            for r in resolved.itertuples():
                eff = r.effective_date
                if not pd.isna(r.old_team_id):
                    conn.execute(
                        "DELETE FROM plyrLogs WHERE player_id = ? AND team_id = ? AND game_date >= ? AND min IS NULL",
                        (r.player_id, r.old_team_id, eff))
                    end = (pd.to_datetime(eff) - pd.Timedelta(days=1)).strftime('%Y-%m-%d')
                    conn.execute("UPDATE rosters SET endDate = ? WHERE teamId = ? AND playerId = ? AND endDate IS NULL",
                                 (end, r.old_team_id, r.player_id))
                if not pd.isna(r.new_team_id):
                    games = conn.execute("SELECT game_id, game_date FROM teamLog WHERE team_id = ? AND game_date >= ?",
                                         (r.new_team_id, max(eff, TODAY))).fetchall()
                    for gid, gdate in games:
                        row = {c: None for c in pcols}
                        row.update({'player_id': r.player_id, 'team_id': r.new_team_id, 'game_id': gid, 'game_date': gdate})
                        conn.execute('INSERT OR IGNORE INTO plyrLogs ({}) VALUES ({})'.format(','.join(pcols), ','.join('?' * len(pcols))),
                                     [row[c] for c in pcols])
                    conn.execute("INSERT OR REPLACE INTO rosters (teamId, seasonYear, playerId, startDate, endDate) VALUES (?, ?, ?, ?, NULL)",
                                 (r.new_team_id, SEASON, r.player_id, eff))
            after = dict(conn.execute(played_q, ids).fetchall())
            if before != after:
                raise RuntimeError('played-game counts changed: before {} after {}'.format(before, after))
        print('Applied. Played-game counts unchanged for all {} players.'.format(len(ids)))
    except Exception as err:
        print('ROLLED BACK, nothing was saved: {}'.format(err))

## 6. Check the result

In [ ]:
for r in resolved.itertuples():
    print('== {}'.format(r.name))
    print(pd.read_sql('''SELECT p.game_date, t.teamAbrv AS team, p.min FROM plyrLogs p LEFT JOIN teams t ON t.team_id = p.team_id
                         WHERE p.player_id = ? AND p.game_date BETWEEN date(?, '-10 days') AND date(?, '+10 days')
                         ORDER BY p.game_date''', conn, params=[r.player_id, r.effective_date, r.effective_date]).to_string(index=False))
    print(pd.read_sql('SELECT t.teamAbrv AS team, r.startDate, r.endDate FROM rosters r LEFT JOIN teams t ON t.team_id = r.teamId WHERE r.playerId = ? ORDER BY r.startDate',
                      conn, params=[r.player_id]).to_string(index=False))

## Undo
The snapshot holds the database exactly as it was before step 5. With the app and scheduled jobs stopped:
```
cp ~/code/bets/nba/data/database/nba_before_roster_change.db ~/code/bets/nba/data/database/nba.db
```
The next run of this notebook overwrites the snapshot, so undo before making another change.